In [10]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json  # 用 json 替代 eval 更安全

# 强制加载 .env 文件（解决加载失败问题）
load_dotenv(dotenv_path="../.env", override=True)
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

model_name = "qwen3.5-flash"

# --------------- 1. 定义工具：计算器 ----------------
def calculator(expression: str) -> str:
    """计算器：计算数学表达式"""
    try:
        result = eval(expression)  # 简单演示，真实场景别直接用eval
        return f"计算结果：{result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

# 工具列表（给LLM看）
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算数学表达式，例如 2+3*5",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，如 100+20*3"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# --------------- 2. Agent 主流程 ----------------
def simple_agent(user_query):
    messages = [
        {"role": "system", "content": "你是一个智能助手，需要计算时必须调用 calculator 工具，不要自己算。"},
        {"role": "user", "content": user_query}
    ]

    # 第一次调用 LLM
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    choice = response.choices[0]
    tool_calls = choice.message.tool_calls

    # 如果 LLM 决定调用工具
    if tool_calls:
        tool_call = tool_calls[0]
        func_name = tool_call.function.name
        args = eval(tool_call.function.arguments)

        print(f"🤖 LLM 决定调用工具：{func_name}，参数：{args}")

        # 执行工具
        if func_name == "calculator":
            tool_result = calculator(args["expression"])

        # 把工具结果放回对话
        messages.append(choice.message)  # 直接把模型返回的工具调用消息加进去
        messages.append({
            "role": "tool",
            "content": tool_result,
            "tool_call_id": tool_call.id
        })

        # LLM 第二次回答：整合结果
        final = client.chat.completions.create(
            model=model_name,
            messages=messages
        )
        return final.choices[0].message.content

    # 不需要工具，直接回答
    else:
        return choice.message.content

# --------------- 测试 ----------------
if __name__ == "__main__":
    question = "3的5次方是多少？再加20等于多少？"
    print("用户问题：", question)
    answer = simple_agent(question)
    print("最终回答：", answer)

用户问题： 3的5次方是多少？再加20等于多少？
🤖 LLM 决定调用工具：calculator，参数：{'expression': '3**5 + 20'}
最终回答： 根据计算结果：

**第一步：** 3的5次方 = 3⁵ = 243  
**第二步：** 243 + 20 = **263**

所以，3的5次方再加20等于 **263**。
